In [ ]:
# Set up the file path
import os
os.chdir('..')

In [ ]:
# Import packages
from RL4CRN.iocrns.reaction_library import construct_hill_production_library, construct_mass_action_library

In [ ]:
species_labels = ['X_1', 'X_2', 'X_3', 'X_4', 'X_5']
n = len(species_labels)
P = 1
R = 4
O = 3
library_hill_production = construct_hill_production_library(species_labels, max_product_order=P, max_num_regulators=R)
library_mass_action = construct_mass_action_library(species_labels, order=O)
print(f"Hill Production Library contains {len(library_hill_production.reactions)} reactions.")
print(f"Mass Action Library contains {len(library_mass_action.reactions)} reactions.")

In [ ]:
import math
# Hill Production reactions count
products_term = math.comb(n + P, P) - 1
regulators_term = sum((1 << t) * math.comb(n, t) for t in range(1, min(R, n) + 1))
print(products_term * regulators_term)

# Mass Action reactions count
Nc = math.comb(n + O, O)
print(Nc * (Nc - 1) + 1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

o = np.arange(1, 4)      # orders 1, 2, 3
n = np.arange(3, 20)

fig, ax = plt.subplots(figsize=(7, 5))

for oi in o:
    vals = []
    for nj in n:
        c = math.comb(nj + oi, oi)
        vals.append(c * (c - 1) + 1)
    ax.plot(n, vals, label=f'Order {oi}')

ax.set_yscale('log')
ax.set_xlabel('Number of Species')
ax.set_ylabel('Number of Reactions')
ax.legend()
ax.set_title('Number of Mass Action Reactions vs Number of Species and Order')

fig.tight_layout()
fig.savefig('num_mass_action_reactions.pdf', bbox_inches='tight')  # save BEFORE show
# plt.show()  # optional in scripts
plt.close(fig)


In [8]:
import numpy as np
import matplotlib.pyplot as plt
import math

# ----------------- Parameters -----------------
n_vals   = np.arange(3, 21)      # x-axis: n
r_values = np.arange(1, 6)       # curves: r = 1..5
p_values = [1, 3]                # one subplot per p
use_logy = True                  # log scale on y-axis
outfile  = "num_mass_action_reactions_vertical.pdf"
# ----------------------------------------------

def binom(a, b):
    if b < 0 or b > a:
        return 0
    return math.comb(a, b)

def sum_term_for_r(n, r):
    """Compute S_r(n) = sum_{i=1..r} 2^i * C(n, i)."""
    s = 0
    for i in range(1, min(r, n) + 1):
        s += (1 << i) * binom(n, i)
    return s

def y_val(n, p, r):
    """Compute y(n; p, r) = ([C(n+p,p)-1]) * Σ_{i=1..r} 2^i * C(n,i)."""
    left = binom(n + p, p) - 1
    right = sum_term_for_r(n, r)
    return left * right

# --- Plot setup ---
fig, axes = plt.subplots(2, 1, figsize=(7, 8), sharex=True, sharey=True)

for ax, p in zip(axes, p_values):
    for r in r_values:
        y = [y_val(int(n), int(p), int(r)) for n in n_vals]
        ax.plot(n_vals, y, linewidth=1.8, label=f"r = {r}")

    ax.set_title(f"p = {p}")
    ax.grid(True, linestyle=":", linewidth=0.7)
    if use_logy:
        ax.set_yscale("log")

axes[-1].set_xlabel("n (number of species)")
axes[0].set_ylabel("Number of reactions")
axes[1].set_ylabel("Number of reactions")

# Place legend only on the bottom plot to avoid repetition
axes[-1].legend(title="r values", loc="upper left")

fig.tight_layout()
fig.savefig(outfile, bbox_inches="tight")
plt.close(fig)

print(f"Saved vertical 2-panel plot to: {outfile}")


Saved vertical 2-panel plot to: num_mass_action_reactions_vertical.pdf
